# 05 - Run Experiments

5x2 cross-validation for the linear, feedforward, and autoencoder networks on each data set.

In [ ]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import load_processed, prepare_xy
from src.neural_network import NeuralNetwork
from src.linear_network import LogisticRegression, LinearRegression
from src.evaluation import cross_validation_loop
from src.tuning import holdout_split, random_search

In [ ]:
seed = 0

### Breast Cancer Dataset

In [ ]:
# Breast Cancer Dataset
data, meta = load_processed("../data/processed/breast_cancer_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [ ]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/breast_cancer_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/breast_cancer_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

### Car Evaluation Dataset

In [ ]:
# Car Evaluation Dataset
data, meta = load_processed("../data/processed/car_evaluation_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [ ]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/car_evaluation_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/car_evaluation_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

### House Votes Dataset

In [ ]:
# House Votes Dataset
data, meta = load_processed("../data/processed/house_votes_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

In [ ]:
# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/house_votes_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": ("int", 2, min(X.shape[1]*2, 32)), "hidden2": ("int", 2, min(X.shape[1]*2, 32)),
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": ("int", 8, max(8, X_cv.shape[0]//10))}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/house_votes_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

### Abalone Dataset

In [ ]:
# Abalone Dataset
data, meta = load_processed("../data/processed/abalone_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [ ]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/abalone_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/abalone_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")

### Computer Hardware Dataset

In [ ]:
# Computer Hardware Dataset
data, meta = load_processed("../data/processed/computer_hardware_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [ ]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/computer_hardware_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/computer_hardware_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")

### Forest Fires Dataset

In [ ]:
# Forest Fires Dataset
data, meta = load_processed("../data/processed/forest_fires_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), None, 0.2, seed=seed)
X_cv, y_cv = X[cv_idx], y[cv_idx]
X_val, y_val = X[val_idx], y[val_idx]

In [ ]:
# ---------------- Linear Regression ----------------
print("Linear Regression")
lin_space = {"learning_rate": ("log_uniform", 0.001, 1.0),
             "batch_size": ("int", 8, max(8, X_cv.shape[0] // 10))}
lin_factory = lambda c, s: LinearRegression(X.shape[1], c["learning_rate"], seed=s)

best_lin, lin_trials = random_search(
    lin_factory, lin_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/forest_fires_linear.csv")
print("Best:", best_lin)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: lin_factory(best_lin, seed), mode="regression",
    epochs=best_lin["epochs"], batch_size=best_lin["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}\n")

In [ ]:
# ---------------- Neural Network ----------------
best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, None, mode="regression",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/forest_fires_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, None, lambda: nn_factory(best_nn, seed), mode="regression",
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Squared Error: {np.mean(results['mse'])}")